# **Feed Forward Network (FFN)**

## What is the Feed-Forward Network (FFN)?

Self-attention lets words talk to each other — it's about relationships. But after that conversation, each word needs time to think individually — to process and digest what it just learned. That's what FFN does.

Think of it like a classroom:
```
Attention = group discussion (students share information)
FFN       = individual thinking (each student processes what they heard)
```

The FFN is applied to each token independently — same formula, same weights, but each word goes through it separately. It doesn't look at other words. That's the key difference from attention.

The formula is simple:
```
FFN(x) = ReLU(x @ W1 + b1) @ W2 + b2
```

That's just two linear layers with a ReLU activation in between.

```
Step 1: x @ W1 + b1        → expand to bigger dimension
Step 2: ReLU()              → kill negative values (non-linearity!)
Step 3: result @ W2 + b2   → compress back to original dimension
```

Why expand then compress?
The standard pattern is to expand to `4x the size`, then compress back:
`d_model = 8  →  expand to 32 (8 × 4)  →  compress back to 8`

```
Like taking a deep breath:
  inhale (expand): give the model MORE space to think, more neurons to work with
  exhale (compress): squeeze the important stuff back into original size
```

This expansion gives the network a bigger "workspace" to compute complex transformations. In real models:

```
GPT-2: d_model=768   → expand to 3072 (768 × 4) → back to 768
GPT-3: d_model=12288 → expand to 49152           → back to 12288
```

Why ReLU?
To solve the `linearity problem`: Two linear layers stacked without activation is just one linear layer — useless stacking. ReLU breaks linearity so the network can learn complex patterns. Without it:
```
(x @ W1) @ W2 = x @ (W1 @ W2) = x @ W_combined
                                  ↑
                        Just ONE linear layer! No benefit from two.
                        
With ReLU between them: can't simplify, genuinely two layers, more power.
```


## What ReLU actually does?

In [ ]:
# Let's see ReLU in action
import torch

sample = torch.tensor([-2.0, -0.5, 0.0, 0.3, 1.5])
print("Before ReLU:", sample)
print("After ReLU: ", torch.relu(sample))
# [-2.0, -0.5, 0.0, 0.3, 1.5]  →  [0.0, 0.0, 0.0, 0.3, 1.5]
# Negatives become zero, positives stay unchanged

Before ReLU: tensor([-2.0000, -0.5000,  0.0000,  0.3000,  1.5000])
After ReLU:  tensor([0.0000, 0.0000, 0.0000, 0.3000, 1.5000])


# FFN - The Manual Way

In [ ]:
import torch
import math

torch.manual_seed(42)

# Fake input: 7 tokens, d_model=8 (pretend this came from attention)
x = torch.randn(7, 8)

d_model = 8
d_ff = 32   # expand to 4x (8 × 4 = 32)

# Create weight matrices manually
W1 = torch.randn(d_model, d_ff)    # (8, 32) — expand
b1 = torch.zeros(d_ff)             # (32,)

W2 = torch.randn(d_ff, d_model)    # (32, 8) — compress back
b2 = torch.zeros(d_model)          # (8,)

# Forward pass — step by step
print("Input shape:", x.shape)                     # (7, 8)

# Step 1: First linear layer (expand)
hidden = x @ W1 + b1
print("After expand:", hidden.shape)                # (7, 32)

# Step 2: ReLU activation (kill negatives)
hidden = torch.relu(hidden)
print("After ReLU:", hidden.shape)                  # (7, 32)
# Some values are now 0 (the negative ones died)

# Step 3: Second linear layer (compress back)
output = hidden @ W2 + b2
print("After compress:", output.shape)              # (7, 8)

print("\nInput shape == Output shape?", x.shape == output.shape)  # True!

Input shape: torch.Size([7, 8])
After expand: torch.Size([7, 32])
After ReLU: torch.Size([7, 32])
After compress: torch.Size([7, 8])

Input shape == Output shape? True


## FFN - PyTorch Way

Same math as the manual version, but PyTorch manages everything. Compare the two:

```
Manual:                          PyTorch:
W1 = torch.randn(8, 32)      →    self.layer1 = nn.Linear(8, 32)
b1 = torch.zeros(32)         →    (bias included automatically)
hidden = x @ W1 + b1         →    x = self.layer1(x)
hidden = torch.relu(hidden)  →    x = self.relu(x)
W2 = torch.randn(32, 8)      →    self.layer2 = nn.Linear(32, 8)
output = hidden @ W2 + b2    →    x = self.layer2(x)
```


In [ ]:
import torch.nn as nn

class FeedForward(nn.Module):
    def __init__(self, d_model, d_ff):
        super().__init__()
        # Two linear layers — expand then compress
        # nn.Linear handles weights AND biases automatically
        self.layer1 = nn.Linear(d_model, d_ff)       # (8 → 32) expand
        self.relu = nn.ReLU()                        # activation
        self.layer2 = nn.Linear(d_ff, d_model)       # (32 → 8) compress

    def forward(self, x):
        # Each step on its own line for clarity
        x = self.layer1(x)    # expand:   (7, 8)  → (7, 32)
        x = self.relu(x)      # activate: (7, 32) → (7, 32) (negatives → 0)
        x = self.layer2(x)    # compress: (7, 32) → (7, 8)
        return x

# Create and run
ffn = FeedForward(d_model=8, d_ff=32)
x = torch.randn(7, 8)
output = ffn(x)

print("Input shape:", x.shape)       # (7, 8)
print("Output shape:", output.shape)  # (7, 8)

# Let's see how many parameters this has
total_params = sum(p.numel() for p in ffn.parameters())
print(f"\nTotal parameters: {total_params}")
# W1: 8×32=256, b1: 32, W2: 32×8=256, b2: 8 → total = 552

Input shape: torch.Size([7, 8])
Output shape: torch.Size([7, 8])

Total parameters: 552
